# Advanced CNN 核心知识点讲解
## 一、基础卷积回顾
卷积神经网络核心分为**特征提取**和**分类**两个阶段，通过卷积层、下采样（Subsampling）层提取特征，再经全连接层（Fully Connected）完成分类，卷积层会改变特征图的通道数和空间尺寸，下采样层主要压缩空间尺寸。

## 二、1×1卷积：核心创新点
### 1. 本质作用
1×1卷积是Advanced CNN的核心基础，**无空间特征提取能力**（仅作用于单个像素），核心作用于**通道维度**：
- 对输入的多通道特征图在每个空间位置做**加权求和**，实现通道的融合、压缩或扩展；
- 每个1×1卷积核对应1个输出通道，卷积核数量=输出通道数，卷积核参数数=输入通道数×1×1。

### 2. 核心优势：减少计算量
以`192@28×28 → 32@28×28`为例，对比两种实现方式的计算量：
- 直接用5×5卷积：$5^2 × 28^2 × 192 × 32 = 120,422,400$
- 1×1卷积+5×5卷积（瓶颈结构）：$1^2 × 28^2 × 192 × 16 + 5^2 × 28^2 × 16 × 32 = 12,433,648$
**通过1×1卷积先压缩通道数，再用大卷积核提取空间特征，可大幅降低计算量**。

## 三、Inception模块（GoogLeNet核心）
### 1. 设计思想
采用**多分支并行**结构，同时用不同尺寸的卷积核（1×1、3×3、5×5）和池化层提取特征，最后将各分支结果**通道维度拼接（Concatenate）**，融合不同尺度的特征信息，提升模型的特征提取能力。

### 2. 经典InceptionA模块结构（各分支均做通道适配）
所有分支先通过1×1卷积调整通道数，再做后续操作，最终拼接输出：
1. **1×1卷积分支**：直接用1×1卷积输出16通道，快速融合通道特征；
2. **5×5卷积分支**：1×1卷积（16通道）→5×5卷积（24通道，padding=2保持空间尺寸）；
3. **双层3×3卷积分支**：1×1卷积（16通道）→3×3卷积（24通道，padding=1）→3×3卷积（24通道，padding=1），替代5×5卷积减少计算；
4. **池化分支**：平均池化（avg_pool2d，3×3，stride=1，padding=1）→1×1卷积（24通道），池化后用1×1卷积调整通道，避免池化后通道数冗余。

### 3. 输出通道数
InceptionA模块各分支输出通道：16+24+24+24 = **88通道**，是固定的拼接结果。

### 4. PyTorch实现要点
- 各分支独立定义卷积层，池化层使用`F.avg_pool2d`并保持空间尺寸；
- 最终通过`torch.cat(outputs, dim=1)`在**通道维度（dim=1）**拼接各分支特征；
- 模块封装为`nn.Module`子类，支持灵活调用。

### 5. 模型效果
基于Inception模块的CNN在手写数字识别任务中，训练10个epoch后**测试集准确率可达98%**，特征融合效果显著。

## 四、残差网络（ResNet）：解决深度网络退化问题
### 1. 核心问题
传统“堆叠卷积层”的深度网络（Plain nets）会出现**退化现象**：网络层数增加，训练误差和测试误差反而升高，原因是深层网络的梯度消失/爆炸，导致参数难以优化。

### 2. 残差思想
引入**残差连接（跳连接）**，将输入$x$直接加到卷积层的输出上，让网络学习**残差函数$F(x)$**，而非直接学习目标函数$H(x)$：
$$H(x) = F(x) + x$$
- 若$F(x)=0$，则$H(x)=x$，网络退化为恒等映射，避免层数增加导致的性能下降；
- 残差连接让梯度可直接通过跳连接回传，缓解梯度消失问题，支持构建更深的网络。

### 3. 基础残差块（Residual Block）实现
- 由两个3×3卷积层组成，均设置`padding=1`，保持输入输出的空间尺寸和通道数一致；
- 卷积后加ReLU激活，**残差相加（x+y）后再做ReLU激活**；
- 封装为`nn.Module`子类，入参为通道数，适配不同通道的特征图。

### 4. 简单残差网络结构
以手写数字识别为例，网络流程：
输入(1,28,28) → 5×5卷积(1→16) → ReLU → 最大池化 → 残差块1(16通道) → 5×5卷积(16→32) → ReLU → 最大池化 → 残差块2(32通道) → 展平 → 全连接层 → 输出(10类)

### 5. 模型效果
残差网络在手写数字识别任务中，训练10个epoch后**测试集准确率可达99%**，比Inception模块模型性能更优，深度网络的训练稳定性大幅提升。

## 五、进阶网络与实践任务
### 1. 残差网络的改进版本
文档给出残差块的改进方向，需结合论文实现：
- 常数缩放（constant scaling）：对残差分支输出做0.5倍缩放后再相加；
- 卷积捷径（conv shortcut）：当输入输出通道数不一致时，用1×1卷积对输入$x$做通道适配，再与$F(x)$相加。

### 2. 稠密连接网络（DenseNet）
作为ResNet的进阶，DenseNet引入**稠密连接**：每个层的输出都与后续所有层的输入拼接，特征复用性更强，文档将其作为实现练习，核心是**BN-ReLU-Conv**的组合层和密集的特征拼接。

### 3. 核心实践要求
- 阅读经典CNN论文（ResNet、DenseNet），理解网络设计思想；
- 基于PyTorch实现改进的残差块和DenseNet，掌握通道适配、特征拼接等核心操作。

## 六、PyTorch实现核心技巧
1. 卷积层设置`padding`保持空间尺寸：$padding = (kernel\_size - 1) / 2$（奇数核）；
2. 池化层使用`stride=1`+`padding`，避免池化导致空间尺寸缩小；
3. 多分支特征融合用`torch.cat(dim=1)`，通道维度拼接是Advanced CNN的核心操作；
4. 自定义模块继承`nn.Module`，在`__init__`定义层，在`forward`实现前向传播；
5. 残差连接要求输入输出的**空间尺寸和通道数完全一致**，否则需做通道/尺寸适配。

## 七、核心结论
1. **1×1卷积**是Advanced CNN的基础，核心用于通道维度操作，减少计算量；
2. **Inception模块**通过多尺度特征融合提升表达能力，核心是**多分支+通道拼接**；
3. **ResNet**通过残差连接解决深度网络退化问题，核心是**学习残差+梯度直传**；
4. 现代CNN的设计趋势是**通道维度的特征融合**和**深层网络的可训练性优化**；
5. PyTorch实现的关键是灵活控制**通道数**和**空间尺寸**，掌握自定义模块和特征拼接。


In [ ]:
# Implementation of Inception Module,Inception模块的实现
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
# 数据预处理
batch_size = 64
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data/mnist/', train=True, download=True,transform=transform)
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_dataset = datasets.MNIST(root='./data/mnist/', train=False, download=True,transform=transform)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)
#inception模块
class Inception(nn.Module):
    def __init__(self,in_chanel):
        super(Inception, self).__init__()
        self.branch1x1 = nn.Conv2d(in_chanel, 16, kernel_size=1)

        self.branch5x5_1 = nn.Conv2d(in_chanel, 16, kernel_size=1)
        self.branch5x5_2 = nn.Conv2d(16, 24, kernel_size=5, padding=2)
        
        self.branch3x3_1 = nn.Conv2d(in_chanel,16,kernel_size=1)
        self.branch3x3_2 = nn.Conv2d(16,24,kernel_size=3,padding=1)
        self.branch3x3_3 = nn.Conv2d(24,24,kernel_size=3,padding=1)

        self.branch_pool = nn.Conv2d(in_chanel, 24, kernel_size=1)

    def forward(self, x):
        branch1x1 = self.branch1x1(x)
        branch5x5 = self.branch5x5_2(self.branch5x5_1(x))
        branch3x3 = self.branch3x3_3(self.branch3x3_2(self.branch3x3_1(x)))
        branch_pool = self.branch_pool(F.avg_pool2d(x, kernel_size=3, stride=1, padding=1))
        return torch.cat([branch1x1, branch5x5, branch3x3, branch_pool], dim=1)
 
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5) 
        self.conv2 = nn.Conv2d(88, 20, kernel_size=5) 
        self.inception1 = Inception(in_chanel = 10)
        self.inception2 = Inception(in_chanel = 20)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(1408, 128)
        self.fc2 = nn.Linear(128, 10)
    def forward(self, x):
        batch_size = x.size(0)
        x = F.relu(self.pool(self.conv1(x)))
        x = self.inception1(x)
        x = F.relu(self.pool(self.conv2(x)))
        x = self.inception2(x)
        x = x.view(batch_size, -1)
        # temp = x.shape
        # print(temp)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = Net()
device = torch.device("xpu" if torch.xpu.is_available() else "cpu")
model.to(device)
criterion = nn.CrossEntropyLoss(reduction='mean')
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.5)

def train(epoch):
    running_loss = 0.0
    model.train()
    for batch_idx, (inputs, target) in enumerate(train_loader):
        inputs,target = inputs.to(device),target.to(device)
        optimizer.zero_grad()
        output = model(inputs)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if batch_idx % 300 == 299:
            print(f"[{epoch+1}, {batch_idx+1}] loss: {running_loss/300:.3f}")
            running_loss = 0.0
def test():
    model.eval()
    total = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, pred = torch.max(output, dim=1) #获取每个样本的预测类别,维度（dim=1）为行，输出value，index
            correct += (pred == target).sum().item()
            total += target.size(0)
    
    print(f"Accuracy in test set: {100. * correct / total:.2f}% [{correct}/{total}]")
if __name__ == '__main__':
    for epoch in range(5):  # 训练5个epoch
        train(epoch)
        test()

[1, 300] loss: 1.544
[1, 600] loss: 0.266
[1, 900] loss: 0.152
Accuracy in test set: 96.49% [9649/10000]
[2, 300] loss: 0.117
[2, 600] loss: 0.094
[2, 900] loss: 0.093
Accuracy in test set: 97.63% [9763/10000]
[3, 300] loss: 0.075
[3, 600] loss: 0.075
[3, 900] loss: 0.064
Accuracy in test set: 98.34% [9834/10000]
[4, 300] loss: 0.061
[4, 600] loss: 0.058
[4, 900] loss: 0.058
Accuracy in test set: 98.29% [9829/10000]
[5, 300] loss: 0.046
[5, 600] loss: 0.050
[5, 900] loss: 0.048
Accuracy in test set: 98.65% [9865/10000]


In [5]:
# Implementation of Simple Residual Network,简单残差网络的实现

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
# 数据预处理
batch_size = 64
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data/mnist/', train=True, download=True,transform=transform)
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_dataset = datasets.MNIST(root='./data/mnist/', train=False, download=True,transform=transform)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

class ResidualBlock(nn.Module):
    def __init__(self,channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        out = F.relu(self.conv1(x))
        out = self.conv2(out)
        return F.relu(out + x) # F(x)+x ,再激活

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,16,kernel_size=5)
        self.conv2 = nn.Conv2d(16,32,kernel_size=5)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(32*4*4, 128)
        self.fc2 = nn.Linear(128, 10)
        self.residual_block1 = ResidualBlock(16)
        self.residual_block2 = ResidualBlock(32)
    
    def forward(self, x):
        batch_size = x.size(0)
        x = self.pool(F.relu(self.conv1(x)))
        x = self.residual_block1(x)
        x = self.pool(F.relu(self.conv2(x)))
        x = self.residual_block2(x)
        # temp = x.shape
        # print(temp)
        x = x.view(batch_size, -1) # 保留批次,展平为1维向量
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x
model = Net()
device = torch.device("xpu" if torch.xpu.is_available() else "cpu")
model.to(device)
criterion = torch.nn.CrossEntropyLoss(reduction='mean')
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.5)

def train(epoch):     
    running_loss = 0.0
    model.train()
    for batch_size,data in enumerate(train_loader):
        inputs,targets = data
        inputs,targets = inputs.to(device),targets.to(device)
        optimizer.zero_grad()
        output = model(inputs)
        loss = criterion(output, targets)
        loss.backward()

        optimizer.step()
        running_loss += loss.item()
        if batch_size % 300 == 299:
            print(f"[{epoch+1}, {batch_size+1}] loss: {running_loss/300:.3f}")
            running_loss = 0.0

def test():
    model.eval()
    currect = 0
    total = 0
    with torch.no_grad():
        for data,targets in test_loader:
            data,targets = data.to(device),targets.to(device)
            output = model(data)
            _,predicted = torch.max(output.data,1)
            total += targets.size(0)
            currect += (predicted == targets).sum().item()
    print(f"Accuracy on test set: {100*currect/total:.2f}% [{currect}/{total}]")

if __name__ == '__main__':
    for epoch in range(10):  # 训练30个epoch
        train(epoch)
        test()
    


[1, 300] loss: 0.723
[1, 600] loss: 0.181
[1, 900] loss: 0.123
Accuracy on test set: 97.29% [9729/10000]
[2, 300] loss: 0.093
[2, 600] loss: 0.080
[2, 900] loss: 0.073
Accuracy on test set: 97.93% [9793/10000]
[3, 300] loss: 0.059
[3, 600] loss: 0.060
[3, 900] loss: 0.055
Accuracy on test set: 98.60% [9860/10000]
[4, 300] loss: 0.047
[4, 600] loss: 0.047
[4, 900] loss: 0.044
Accuracy on test set: 98.72% [9872/10000]
[5, 300] loss: 0.039
[5, 600] loss: 0.039
[5, 900] loss: 0.038
Accuracy on test set: 98.81% [9881/10000]
[6, 300] loss: 0.032
[6, 600] loss: 0.031
[6, 900] loss: 0.035
Accuracy on test set: 99.00% [9900/10000]
[7, 300] loss: 0.029
[7, 600] loss: 0.027
[7, 900] loss: 0.029
Accuracy on test set: 98.82% [9882/10000]
[8, 300] loss: 0.024
[8, 600] loss: 0.027
[8, 900] loss: 0.024
Accuracy on test set: 98.72% [9872/10000]
[9, 300] loss: 0.023
[9, 600] loss: 0.020
[9, 900] loss: 0.023
Accuracy on test set: 98.29% [9829/10000]
[10, 300] loss: 0.019
[10, 600] loss: 0.019
[10, 900] l

In [ ]:
# constant scaling,常量缩放
class ResidualBlock(nn.Module):
    def __init__(self,channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        out = F.relu(self.conv1(x))
        out = self.conv2(out)
        return F.relu((out + x)/2) # (F(x)+x)*0.5,再激活

In [ ]:
# conv shortcut,卷积快捷连接
class ResidualBlock(nn.Module):
    def __init__(self,channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.conv1x1 = nn.Conv2d(channels, channels, kernel_size=1, padding=0)
        
    def forward(self, x):
        out = F.relu(self.conv1(x))
        out = self.conv2(out)
        out1 = self.conv1x1(x)
        return F.relu((out + out1)) 
